<a href="https://colab.research.google.com/github/akorzun159/cohort-ltv-analysis/blob/main/cohort-ltv-analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Техническое задание: Когортный анализ LTV**

**Бизнес-контекст**

Компания занимается онлайн-продажей электроники. Маркетинговая команда хочет понять, как меняется ценность клиентов в зависимости от времени их привлечения, товарной категории, с которой они начинали, и платформы первой покупки. Это поможет оценить эффективность маркетинговых кампаний по месяцам, определить, какие категории товаров привлекают самых ценных клиентов, и выявить, с каких платформ приходят самые доходные пользователи.


**Описание данных**

Таблица orders содержит информацию о заказах: order_id — уникальный идентификатор заказа, customer_id — уникальный идентификатор клиента, order_date — дата создания заказа, order_amount — сумма заказа, platform — платформа заказа (mobile/desktop/app), category — категория товара.


**Постановка задачи**

Построить когортный анализ, в котором когорта определяется месяцем первой покупки клиента. Клиент приписывается только к одной когорте — месяцу его самого первого заказа. Необходимо рассчитать средний доход с клиента (LTV) за первые 10 дней после первой покупки. Доход считается только за первые 10 дней (включительно: день 0 + 9 следующих дней), все клиенты в когорте имеют одинаковое окно для анализа. Срезы данных выполняются по категории товара из первого заказа клиента и по платформе из первого заказа клиента. Итоговый результат должен содержать следующие поля: cohort_month — месяц первой покупки клиента в формате YYYY-MM, first_category — категория товара из первого заказа, first_platform — платформа из первого заказа, cohort_size — количество клиентов в когорте, avg_revenue — средний доход на клиента за первые 10 дней.

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

Загрузим таблицу 'orders.csv'

In [2]:
orders = pd.read_csv('/content/drive/MyDrive/orders_10000.csv')

Посмотри первые 5 строк и типы данных в таблице 'orders.csv'

In [3]:
orders.head()

,order_id,customer_id,order_date,order_amount,platform,category
0,1,1,2023-04-13,11543.21,app,смартфоны
1,2,1,2023-09-20,60374.30,mobile,наушники
2,3,2,2024-03-11,312233.66,desktop,планшеты
3,5,2,2024-04-07,50270.29,mobile,ноутбуки
4,6,2,2024-05-09,52314.64,app,смартфоны


In [4]:
orders.dtypes

,0
order_id,int64
customer_id,int64
order_date,object
order_amount,float64
platform,object
category,object


Как можно увидеть столбец order_date имеет тип данных object. Преобразуем его в datetime64 для работы с датами

In [5]:
orders['order_date'] = pd.to_datetime(orders['order_date'])

In [6]:
orders.dtypes

,0
order_id,int64
customer_id,int64
order_date,datetime64[ns]
order_amount,float64
platform,object
category,object


Преобразование выполнено успешно. Далее выделим дату первого заказа, категорию товаров и платформу, на которой клиент впервые сделал заказ

In [7]:
first_order = orders[['customer_id', 'platform', 'category', 'order_date']].sort_values('order_date')\
    .groupby('customer_id', as_index=False)\
    .first()\
    .reset_index(drop=True)\
    .rename(columns={
        'platform': 'first_platform',
        'category': 'first_category',
        'order_date': 'first_order_date'
    })

In [8]:
first_order

,customer_id,first_platform,first_category,first_order_date
0,1,app,смартфоны,2023-04-13
1,2,desktop,планшеты,2024-03-11
2,3,desktop,ноутбуки,2023-10-13
3,4,desktop,смартфоны,2023-04-17
4,5,desktop,планшеты,2023-03-13
...,...,...,...,...
2807,2853,desktop,смартфоны,2024-09-17
2808,2854,mobile,ноутбуки,2023-02-09
2809,2855,desktop,наушники,2024-08-12
2810,2856,mobile,планшеты,2023-08-08


Определим когортный месяц (месяц, когда клиент впервые сделал заказ) для каждого клиента

In [9]:
first_order['cohort_month'] = first_order['first_order_date'].dt.to_period('M')

In [10]:
first_order

,customer_id,first_platform,first_category,first_order_date,cohort_month
0,1,app,смартфоны,2023-04-13,2023-04
1,2,desktop,планшеты,2024-03-11,2024-03
2,3,desktop,ноутбуки,2023-10-13,2023-10
3,4,desktop,смартфоны,2023-04-17,2023-04
4,5,desktop,планшеты,2023-03-13,2023-03
...,...,...,...,...,...
2807,2853,desktop,смартфоны,2024-09-17,2024-09
2808,2854,mobile,ноутбуки,2023-02-09,2023-02
2809,2855,desktop,наушники,2024-08-12,2024-08
2810,2856,mobile,планшеты,2023-08-08,2023-08


Объединим исходную таблицу order с полученной нами таблицей first_order


In [11]:
order_with_first_indicators = orders.merge(first_order, on='customer_id')

In [12]:
order_with_first_indicators


,order_id,customer_id,order_date,order_amount,platform,category,first_platform,first_category,first_order_date,cohort_month
0,1,1,2023-04-13,11543.21,app,смартфоны,app,смартфоны,2023-04-13,2023-04
1,2,1,2023-09-20,60374.30,mobile,наушники,app,смартфоны,2023-04-13,2023-04
2,3,2,2024-03-11,312233.66,desktop,планшеты,desktop,планшеты,2024-03-11,2024-03
3,5,2,2024-04-07,50270.29,mobile,ноутбуки,desktop,планшеты,2024-03-11,2024-03
4,6,2,2024-05-09,52314.64,app,смартфоны,desktop,планшеты,2024-03-11,2024-03
...,...,...,...,...,...,...,...,...,...,...
9995,12902,2856,2023-10-01,58138.75,mobile,смартфоны,mobile,планшеты,2023-08-08,2023-08
9996,12903,2856,2024-01-10,42333.47,mobile,смартфоны,mobile,планшеты,2023-08-08,2023-08
9997,12904,2857,2023-05-09,19938.93,mobile,ноутбуки,mobile,ноутбуки,2023-05-09,2023-05
9998,12905,2857,2023-06-18,110394.37,mobile,ноутбуки,mobile,ноутбуки,2023-05-09,2023-05


Далее посчитаем количество дней между первым заказом клиентов и всеми последующими

In [13]:
order_with_first_indicators['days_since_first_order'] = (order_with_first_indicators['order_date']-\
                                                         order_with_first_indicators['first_order_date']).dt.days

In [14]:
order_with_first_indicators

,order_id,customer_id,order_date,order_amount,platform,category,first_platform,first_category,first_order_date,cohort_month,days_since_first_order
0,1,1,2023-04-13,11543.21,app,смартфоны,app,смартфоны,2023-04-13,2023-04,0
1,2,1,2023-09-20,60374.30,mobile,наушники,app,смартфоны,2023-04-13,2023-04,160
2,3,2,2024-03-11,312233.66,desktop,планшеты,desktop,планшеты,2024-03-11,2024-03,0
3,5,2,2024-04-07,50270.29,mobile,ноутбуки,desktop,планшеты,2024-03-11,2024-03,27
4,6,2,2024-05-09,52314.64,app,смартфоны,desktop,планшеты,2024-03-11,2024-03,59
...,...,...,...,...,...,...,...,...,...,...,...
9995,12902,2856,2023-10-01,58138.75,mobile,смартфоны,mobile,планшеты,2023-08-08,2023-08,54
9996,12903,2856,2024-01-10,42333.47,mobile,смартфоны,mobile,планшеты,2023-08-08,2023-08,155
9997,12904,2857,2023-05-09,19938.93,mobile,ноутбуки,mobile,ноутбуки,2023-05-09,2023-05,0
9998,12905,2857,2023-06-18,110394.37,mobile,ноутбуки,mobile,ноутбуки,2023-05-09,2023-05,40


Теперь выберем только те заказы, которые были совершены не более, чем за 10 дней после первой покупки. Используем оператор < 10, так как нам нужно первые 10 дней: 0,1,...,9

In [15]:
order_less_than_10_days = order_with_first_indicators.loc[
    order_with_first_indicators['days_since_first_order'] < 10,
    ['order_id', 'customer_id', 'order_date', 'order_amount',
     'first_platform', 'first_category', 'first_order_date',
     'cohort_month', 'days_since_first_order']
]

In [16]:
order_less_than_10_days

,order_id,customer_id,order_date,order_amount,first_platform,first_category,first_order_date,cohort_month,days_since_first_order
0,1,1,2023-04-13,11543.21,app,смартфоны,2023-04-13,2023-04,0
2,3,2,2024-03-11,312233.66,desktop,планшеты,2024-03-11,2024-03,0
9,12,3,2023-10-13,51884.30,desktop,ноутбуки,2023-10-13,2023-10,0
11,17,4,2023-04-17,7507.08,desktop,смартфоны,2023-04-17,2023-04,0
14,21,5,2023-03-13,4720.90,desktop,планшеты,2023-03-13,2023-03,0
...,...,...,...,...,...,...,...,...,...
9984,12890,2854,2023-02-09,39404.82,mobile,ноутбуки,2023-02-09,2023-02,0
9989,12896,2855,2024-08-12,72714.58,desktop,наушники,2024-08-12,2024-08,0
9990,12897,2856,2023-08-08,10894.21,mobile,планшеты,2023-08-08,2023-08,0
9991,12898,2856,2023-08-16,49919.25,mobile,планшеты,2023-08-08,2023-08,8


Для каждого клиента определим общую сумму заказов за первые 10 дней включительно.

In [17]:
total_amount = order_less_than_10_days.groupby('customer_id', as_index=False).agg({'order_amount': 'sum'}).rename(columns={'order_amount':'total_amount'})

In [18]:
total_amount

,customer_id,total_amount
0,1,11543.21
1,2,312233.66
2,3,51884.30
3,4,7507.08
4,5,73178.07
...,...,...
2807,2853,6645.37
2808,2854,39404.82
2809,2855,72714.58
2810,2856,60813.46


Объединим таблицы order_less_than_10_days и total_amount

In [19]:
order_less_than_10_days = order_less_than_10_days.merge(total_amount, on='customer_id')

In [20]:
order_less_than_10_days

,order_id,customer_id,order_date,order_amount,first_platform,first_category,first_order_date,cohort_month,days_since_first_order,total_amount
0,1,1,2023-04-13,11543.21,app,смартфоны,2023-04-13,2023-04,0,11543.21
1,3,2,2024-03-11,312233.66,desktop,планшеты,2024-03-11,2024-03,0,312233.66
2,12,3,2023-10-13,51884.30,desktop,ноутбуки,2023-10-13,2023-10,0,51884.30
3,17,4,2023-04-17,7507.08,desktop,смартфоны,2023-04-17,2023-04,0,7507.08
4,21,5,2023-03-13,4720.90,desktop,планшеты,2023-03-13,2023-03,0,73178.07
...,...,...,...,...,...,...,...,...,...,...
3218,12890,2854,2023-02-09,39404.82,mobile,ноутбуки,2023-02-09,2023-02,0,39404.82
3219,12896,2855,2024-08-12,72714.58,desktop,наушники,2024-08-12,2024-08,0,72714.58
3220,12897,2856,2023-08-08,10894.21,mobile,планшеты,2023-08-08,2023-08,0,60813.46
3221,12898,2856,2023-08-16,49919.25,mobile,планшеты,2023-08-08,2023-08,8,60813.46


Далее выделим только те столбцы, которые пригодятся нам в дальнейшем исследовании, и выделим только уникальные значения

In [21]:
unique_customers = order_less_than_10_days[['customer_id', 'first_platform', 'first_category', 'first_order_date', 'cohort_month', 'total_amount']]\
.drop_duplicates(subset='customer_id', keep='first')

In [22]:
unique_customers

,customer_id,first_platform,first_category,first_order_date,cohort_month,total_amount
0,1,app,смартфоны,2023-04-13,2023-04,11543.21
1,2,desktop,планшеты,2024-03-11,2024-03,312233.66
2,3,desktop,ноутбуки,2023-10-13,2023-10,51884.30
3,4,desktop,смартфоны,2023-04-17,2023-04,7507.08
4,5,desktop,планшеты,2023-03-13,2023-03,73178.07
...,...,...,...,...,...,...
3217,2853,desktop,смартфоны,2024-09-17,2024-09,6645.37
3218,2854,mobile,ноутбуки,2023-02-09,2023-02,39404.82
3219,2855,desktop,наушники,2024-08-12,2024-08,72714.58
3220,2856,mobile,планшеты,2023-08-08,2023-08,60813.46


Последним шагом сделаем группировки по cohort_month first_category и first_platform. Посчитаем средний доход для каждой когорты и определим количество покупателей, которые входят в одну когорту

In [23]:
cohort_result = unique_customers.groupby(['cohort_month', 'first_category', 'first_platform'])\
.agg(\
     avg_revenue=('total_amount', 'mean'), \
     cohort_size=('customer_id', 'nunique'))\
.reset_index()

In [24]:
cohort_result

,cohort_month,first_category,first_platform,avg_revenue,cohort_size
0,2023-01,аксессуары,app,684278.404286,7
1,2023-01,аксессуары,desktop,31672.900000,4
2,2023-01,аксессуары,mobile,57730.480000,2
3,2023-01,наушники,app,88336.640000,5
4,2023-01,наушники,desktop,58690.508000,5
...,...,...,...,...,...
353,2024-12,планшеты,desktop,42344.525000,2
354,2024-12,планшеты,mobile,110978.442500,4
355,2024-12,смартфоны,app,117681.148000,5
356,2024-12,смартфоны,desktop,144314.531111,9


Для того, чтобы работать с датой было проще, например, при работе в Excel в будущем, переведем тип данных datetime64 в str

In [25]:
cohort_result['cohort_month'] = cohort_result['cohort_month'].astype(str)

In [26]:
cohort_result.dtypes

,0
cohort_month,object
first_category,object
first_platform,object
avg_revenue,float64
cohort_size,int64


Итоговая таблица:

In [27]:
cohort_result

,cohort_month,first_category,first_platform,avg_revenue,cohort_size
0,2023-01,аксессуары,app,684278.404286,7
1,2023-01,аксессуары,desktop,31672.900000,4
2,2023-01,аксессуары,mobile,57730.480000,2
3,2023-01,наушники,app,88336.640000,5
4,2023-01,наушники,desktop,58690.508000,5
...,...,...,...,...,...
353,2024-12,планшеты,desktop,42344.525000,2
354,2024-12,планшеты,mobile,110978.442500,4
355,2024-12,смартфоны,app,117681.148000,5
356,2024-12,смартфоны,desktop,144314.531111,9
